# This code takes in the upset frequency files, and creates a CSV of the upset frequency per season per league.

In [ ]:
# --- Build upset-frequency tables from game-level CSVs ---
# INPUTS
IN_COMBINED = "/mnt/data/us_combined_with_upset.csv"               # real data
IN_SKILL    = "/mnt/data/us_pure_skill_with_upset.csv"            # pure-skill
IN_LUCK     = "/mnt/data/us_combined_coinflip_0.5_with_upset.csv" # coin-flip (luck)

# OUTPUTS
OUT_COMBINED = "/mnt/data/us_combined_upset_frequency.csv"
OUT_SKILL    = "/mnt/data/us_pure_skill_upset_frequency.csv"
OUT_LUCK     = "/mnt/data/us_coinflip_0.5_upset_frequency.csv"

import pandas as pd
import numpy as np

def _detect_cols(path):
    hdr = pd.read_csv(path, nrows=0)
    raw = list(hdr.columns)
    cols = [c.strip() for c in raw]
    lower = [c.lower() for c in cols]

    def find_one(cands):
        for c in cands:
            if c in lower:
                return cols[lower.index(c)]
        return None

    league = find_one(["league","competition","sport"])
    season = find_one(["season","year","season_year","seas"])
    upset  = None
    for exact in ["upset","is_upset","was_upset","upset_flag"]:
        if exact in lower:
            upset = cols[lower.index(exact)]
            break
    if upset is None:
        for i, c in enumerate(lower):
            if "upset" in c:
                upset = cols[i]; break

    if league is None or season is None or upset is None:
        raise ValueError(
            f"Could not detect columns in {path}. "
            f"Detected -> league: {league}, season: {season}, upset: {upset}"
        )
    return league, season, upset

def _to01(series):
    if series.dtype == bool:
        return series.astype(int).astype(float)
    # try numeric
    num = pd.to_numeric(series, errors="coerce")
    if num.notna().any():
        return (num > 0).astype(float)
    # fall back to strings
    s = series.astype(str).str.strip().str.lower()
    return s.isin({"1","true","t","yes","y","upset"}).astype(float)

def build_upset_frequency(input_path, output_path, chunksize=250_000):
    league_col, season_col, upset_col = _detect_cols(input_path)

    totals = {}       # {(league, season): total_games}
    upset_totals = {} # {(league, season): upset_games}

    usecols = [league_col, season_col, upset_col]
    for chunk in pd.read_csv(input_path, usecols=usecols, chunksize=chunksize):
        chunk = chunk.rename(columns={
            league_col: "league",
            season_col: "season",
            upset_col:  "upset"
        })
        chunk["upset"] = _to01(chunk["upset"])

        grp = chunk.groupby(["league","season"], observed=True)["upset"]
        cnt = grp.size()
        ups = grp.sum()

        # accumulate
        for key, n in cnt.items():
            totals[key] = totals.get(key, 0) + int(n)
        for key, u in ups.items():
            upset_totals[key] = upset_totals.get(key, 0.0) + float(u)

    rows = []
    for key, total in totals.items():
        upsum = upset_totals.get(key, 0.0)
        freq = (upsum / total) if total else np.nan
        rows.append({"league": key[0], "season": key[1], "upset_frequency": freq})

    out = pd.DataFrame(rows).sort_values(["league","season"]).reset_index(drop=True)
    out.to_csv(output_path, index=False)
    print(f"Saved {len(out):,} rows -> {output_path}")
    return out

# Run for all three datasets
_ = build_upset_frequency(IN_COMBINED, OUT_COMBINED)
_ = build_upset_frequency(IN_SKILL,    OUT_SKILL)
_ = build_upset_frequency(IN_LUCK,     OUT_LUCK)


Saved 180 rows -> /mnt/data/us_combined_upset_frequency.csv
Saved 180 rows -> /mnt/data/us_pure_skill_upset_frequency.csv
Saved 180 rows -> /mnt/data/us_coinflip_0.5_upset_frequency.csv
